# RDDs en Apache Spark para Big Data

Notebook único (todas las secciones unidas) con ejemplos ejecutables sobre PySpark en modo local. Cada sección abre y cierra su propia `SparkSession` para poder ejecutarse de forma independiente si se corren las celdas por partes.

# 1. ¿Qué es un RDD?

Un **RDD** (*Resilient Distributed Dataset*) es la estructura de datos
fundamental de Apache Spark. Es la abstracción original sobre la que
se construyeron después los DataFrames y Datasets.

Un RDD es:

- **Resiliente**: si un nodo del clúster falla, Spark puede reconstruir
  la parte perdida del RDD usando su *linaje* (el historial de
  transformaciones que lo generaron), sin necesidad de replicar los
  datos en disco constantemente.
- **Distribuido**: los datos de un RDD están divididos en **particiones**
  que viven en distintos nodos (ejecutores) del clúster y se procesan
  en paralelo.
- **Dataset**: una colección de elementos (pueden ser números, tuplas,
  objetos, líneas de texto, etc.) sin un esquema tabular obligatorio,
  a diferencia de un DataFrame.

## Propiedades clave

1. **Inmutable**: un RDD nunca se modifica; cada transformación crea
   un RDD nuevo.
2. **Evaluación perezosa (*lazy evaluation*)**: las transformaciones
   (`map`, `filter`, etc.) no se ejecutan de inmediato. Spark solo
   construye un **grafo de ejecución (DAG)** y lo ejecuta cuando se
   invoca una **acción** (`collect`, `count`, `saveAsTextFile`, ...).
3. **Tolerante a fallos por linaje**: en vez de guardar copias de los
   datos, Spark guarda la secuencia de operaciones para poder
   recomputar una partición perdida.

## ¿Por qué siguen siendo importantes si ya existen los DataFrames?

Los DataFrames (con el optimizador **Catalyst**) son más rápidos y
fáciles de usar para datos tabulares, pero los RDDs siguen siendo la
base interna de Spark y son útiles cuando:

- Se necesita control fino sobre el particionamiento físico.
- Los datos no tienen estructura tabular (grafos, texto libre,
  objetos complejos, algoritmos científicos).
- Se requiere lógica de transformación arbitraria que no se expresa
  bien con funciones SQL.

En el último notebook de este libro (RDD vs DataFrame) profundizamos
en cuándo conviene cada uno.

## Creando una SparkSession

Toda aplicación Spark moderna arranca creando una `SparkSession`. De
ahí se obtiene el `SparkContext` (`sc`), que es el objeto clásico con
el que se crean y manipulan RDDs.

In [12]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("01-que-es-rdd")
    .master("local[*]")          # usa todos los núcleos disponibles en esta máquina
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")  # silenciar logs ruidosos de Spark

sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=02-creacion-rdds>

`local[*]` le dice a Spark que simule un clúster usando los núcleos
de esta máquina como si fueran "ejecutores". En un clúster real,
`master` apuntaría a un *cluster manager* como YARN, Kubernetes o el
standalone de Spark (por ejemplo `spark://host:7077`).

## Nuestro primer RDD

In [13]:
numeros = sc.parallelize(range(1, 11))
print("Tipo de objeto:", type(numeros))
print("Número de particiones:", numeros.getNumPartitions())

Tipo de objeto: <class 'pyspark.core.rdd.PipelinedRDD'>
Número de particiones: 2


Nótese que hasta este punto **no se ha ejecutado ningún cómputo real**.
`parallelize` es una transformación: Spark solo anotó "este RDD se
construye distribuyendo esta lista de Python". Al llamar a una acción
como `collect()`, recién ahí se dispara el trabajo.

In [14]:
resultado = numeros.collect()
print(resultado)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## El DAG y la evaluación perezosa

Encadenemos dos transformaciones y veamos que nada se ejecuta hasta
que llamamos a una acción.

In [15]:
pares = numeros.filter(lambda x: x % 2 == 0)
cuadrados = pares.map(lambda x: x ** 2)

print("¿Ya se calculó algo? No. `cuadrados` es solo un plan:")
print(cuadrados)

¿Ya se calculó algo? No. `cuadrados` es solo un plan:
PythonRDD[9] at RDD at PythonRDD.scala:57


In [16]:
# Aquí sí se dispara el cómputo: filter -> map -> collect
print("Resultado real:", cuadrados.collect())

Resultado real: [4, 16, 36, 64, 100]


Podemos inspeccionar el plan de ejecución (el DAG) con `toDebugString()`:

In [17]:
print(cuadrados.toDebugString().decode("utf-8"))

(2) PythonRDD[9] at RDD at PythonRDD.scala:57 []
 |  ParallelCollectionRDD[7] at readRDDFromFile at PythonRDD.scala:298 []


Cada línea representa una etapa del linaje: el `PythonRDD` final
depende del `MapPartitionsRDD` del `filter`, que a su vez depende del
RDD generado por `parallelize`. Si un ejecutor perdiera la partición
del resultado final, Spark sabe exactamente cómo reconstruirla
repitiendo estos pasos.

## Cerrar la sesión

Es buena práctica detener la `SparkSession` cuando terminamos, para
liberar recursos (en este libro cada notebook abre y cierra la suya).

In [18]:
spark.stop()

---

# 2. Formas de crear RDDs

Hay tres maneras principales de obtener un RDD:

1. Paralelizando una colección de Python ya existente en memoria
   (`sc.parallelize`).
2. Leyendo datos externos (archivos de texto, CSV, carpetas completas).
3. Transformando un RDD existente (esto se ve en el notebook de
   transformaciones).

In [19]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("02-creacion-rdds")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

## 2.1 Desde una colección en memoria: `parallelize`

In [20]:
ciudades = sc.parallelize(["Manizales", "Bogota", "Medellin", "Cali"])
print(ciudades.collect())

['Manizales', 'Bogota', 'Medellin', 'Cali']


El segundo argumento de `parallelize` controla cuántas particiones se
crean. Esto importa para el nivel de paralelismo real del trabajo.

In [21]:
numeros_4p = sc.parallelize(range(1, 21), numSlices=4)
print("Particiones:", numeros_4p.getNumPartitions())

# glom() agrupa cada partición en una lista, útil solo para inspeccionar
for i, particion in enumerate(numeros_4p.glom().collect()):
    print(f"Partición {i}: {particion}")

Particiones: 4
Partición 0: [1, 2, 3, 4, 5]
Partición 1: [6, 7, 8, 9, 10]
Partición 2: [11, 12, 13, 14, 15]
Partición 3: [16, 17, 18, 19, 20]


## 2.2 Desde un archivo de texto: `textFile`

`textFile` crea un RDD donde **cada línea del archivo es un
elemento** del RDD (un string).

In [22]:
lineas = sc.textFile("quijote_fragmento.txt")
print("Número de líneas:", lineas.count())
print("Primera línea:", lineas.first())

Número de líneas: 12
Primera línea: En un lugar de la Mancha, de cuyo nombre no quiero acordarme, no ha mucho


`textFile` también acepta:

- Una carpeta completa (lee todos los archivos dentro).
- Patrones glob, ej: `sc.textFile("carpeta/*.txt")`.
- Rutas de HDFS, S3, Azure Blob, etc. (`hdfs://...`, `s3a://...`),
  exactamente igual que una ruta local — Spark abstrae el sistema de
  archivos subyacente.

## 2.3 Desde un CSV como texto plano

`textFile` no interpreta comas ni encabezados: solo entrega líneas
crudas. Para RDDs, el parseo de columnas es responsabilidad nuestra
(por eso, para datos tabulares, casi siempre conviene un DataFrame en
vez de un RDD — ver el último notebook del libro).

In [23]:
ventas_raw = sc.textFile("ventas.csv")
encabezado = ventas_raw.first()
print("Encabezado:", encabezado)

# quitamos el encabezado y separamos por comas manualmente
ventas_filas = (
    ventas_raw
    .filter(lambda linea: linea != encabezado)
    .map(lambda linea: linea.split(","))
)
ventas_filas.take(3)

Encabezado: fecha,ciudad,producto,categoria,unidades,precio_unitario


[['2026-01-05', 'Manizales', 'Cafe Premium', 'Bebidas', '120', '18500'],
 ['2026-01-05', 'Bogota', 'Cafe Premium', 'Bebidas', '300', '18500'],
 ['2026-01-06', 'Medellin', 'Arepa Congelada', 'Alimentos', '80', '4200']]

## 2.4 `wholeTextFiles`: cuando el archivo completo es la unidad

A diferencia de `textFile` (una línea = un elemento), `wholeTextFiles`
entrega pares `(nombre_de_archivo, contenido_completo)`. Es útil
cuando cada archivo representa un documento entero (por ejemplo,
muchos archivos pequeños de log o de texto).

In [24]:
documentos = sc.wholeTextFiles("data/")
for nombre, contenido in documentos.collect():
    print(nombre, "->", len(contenido), "caracteres")

In [25]:
spark.stop()

---

# 3. Transformaciones

Las **transformaciones** son operaciones que reciben un RDD y
devuelven **otro RDD nuevo**. Nunca modifican el original (los RDDs
son inmutables) y son **perezosas**: no se ejecutan hasta que una
acción las dispara.

In [26]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("03-transformaciones")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

numeros = sc.parallelize(range(1, 11))
palabras = sc.parallelize(["spark es rapido", "los rdd son inmutables", "spark procesa en paralelo"])

## `map`: transforma cada elemento 1 a 1

In [27]:
al_cuadrado = numeros.map(lambda x: x ** 2)
print(al_cuadrado.collect())

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]


## `flatMap`: transforma y "aplana" el resultado

`map` produce exactamente un elemento de salida por cada elemento de
entrada. `flatMap` produce **cero o más** elementos de salida por
cada entrada, y los aplana en un solo RDD plano. Es el operador
clásico para tokenizar texto en palabras.

In [28]:
solo_map = palabras.map(lambda linea: linea.split(" "))
print("Con map (lista de listas):", solo_map.collect())

con_flatmap = palabras.flatMap(lambda linea: linea.split(" "))
print("Con flatMap (lista plana):", con_flatmap.collect())

Con map (lista de listas): [['spark', 'es', 'rapido'], ['los', 'rdd', 'son', 'inmutables'], ['spark', 'procesa', 'en', 'paralelo']]
Con flatMap (lista plana): ['spark', 'es', 'rapido', 'los', 'rdd', 'son', 'inmutables', 'spark', 'procesa', 'en', 'paralelo']


## `filter`: se queda solo con los elementos que cumplen una condición

In [29]:
pares = numeros.filter(lambda x: x % 2 == 0)
print(pares.collect())

[2, 4, 6, 8, 10]


## `distinct`: elimina duplicados

In [30]:
con_repetidos = sc.parallelize([1, 2, 2, 3, 3, 3, 4])
print(con_repetidos.distinct().collect())

[2, 4, 1, 3]


## `union` e `intersection`: combinar RDDs

In [31]:
a = sc.parallelize([1, 2, 3, 4])
b = sc.parallelize([3, 4, 5, 6])

print("Unión:", sorted(a.union(b).collect()))
print("Intersección:", sorted(a.intersection(b).collect()))
print("Diferencia (a - b):", sorted(a.subtract(b).collect()))

Unión: [1, 2, 3, 3, 4, 4, 5, 6]
Intersección: [3, 4]
Diferencia (a - b): [1, 2]


## `sample`: tomar una muestra aleatoria

In [32]:
grande = sc.parallelize(range(1, 1001))
muestra = grande.sample(withReplacement=False, fraction=0.01, seed=42)
print("Tamaño de la muestra:", muestra.count())
print("Algunos elementos:", muestra.take(5))

Tamaño de la muestra: 13
Algunos elementos: [11, 116, 261, 282, 289]


## Encadenando transformaciones (pipeline perezoso)

Como cada transformación devuelve un RDD, se pueden encadenar en una
sola expresión. Spark construye el DAG completo y solo lo ejecuta al
final, cuando llamamos `collect()`.

In [33]:
resultado = (
    palabras
    .flatMap(lambda linea: linea.split(" "))
    .filter(lambda palabra: len(palabra) > 3)
    .map(lambda palabra: palabra.upper())
    .distinct()
)
print(resultado.collect())

['PARALELO', 'SPARK', 'RAPIDO', 'INMUTABLES', 'PROCESA']


## `mapPartitions`: transformar por partición completa, no elemento a elemento

A veces el costo de inicializar algo (una conexión, un modelo cargado)
es alto y no queremos pagarlo por cada elemento, sino una vez por
partición. `mapPartitions` recibe un iterador con todos los elementos
de la partición y debe devolver un iterador.

In [34]:
def procesar_particion(iterador):
    # simula, p. ej., abrir una conexión costosa UNA sola vez por partición
    total_particion = 0
    conteo = 0
    for valor in iterador:
        total_particion += valor
        conteo += 1
    yield (conteo, total_particion)

resumen_por_particion = numeros.mapPartitions(procesar_particion)
print(resumen_por_particion.collect())

[(5, 15), (5, 40)]


In [35]:
spark.stop()

---

# 4. Acciones

Las **acciones** son las operaciones que realmente disparan el
cómputo sobre el DAG construido por las transformaciones, y devuelven
un resultado al programa Python (el *driver*), o lo escriben a un
almacenamiento externo.

Es la diferencia clave con las transformaciones:

| | Entrada | Salida | ¿Ejecuta de inmediato? |
|---|---|---|---|
| Transformación | RDD | RDD | No (perezosa) |
| Acción | RDD | Valor en el driver / archivo | Sí |

In [36]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("04-acciones")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

numeros = sc.parallelize(range(1, 101), numSlices=8)

## `collect`: trae TODOS los elementos al driver

⚠️ Peligroso con datasets grandes: si el RDD tiene millones de
elementos distribuidos en el clúster, `collect()` intenta traerlos
todos a la memoria de una sola máquina (el driver) y puede tumbarla.
Úsese solo cuando se sabe que el resultado es pequeño, o para
depuración con muestras.

In [37]:
pequeno = sc.parallelize([1, 2, 3])
print(pequeno.collect())

[1, 2, 3]


## `count`: cuenta elementos

In [38]:
print("Total de elementos:", numeros.count())

Total de elementos: 100


## `first` y `take(n)`: traer solo una parte

In [39]:
print("Primero:", numeros.first())
print("Primeros 5:", numeros.take(5))

Primero: 1
Primeros 5: [1, 2, 3, 4, 5]


## `reduce`: combina todos los elementos en uno solo

Recibe una función binaria asociativa y conmutativa (importante: en
un entorno distribuido no se controla el orden exacto en que se
combinan las particiones).

In [40]:
suma_total = numeros.reduce(lambda a, b: a + b)
print("Suma con reduce:", suma_total)

maximo = numeros.reduce(lambda a, b: a if a > b else b)
print("Máximo con reduce:", maximo)

Suma con reduce: 5050
Máximo con reduce: 100


## `fold`: como `reduce`, pero con un valor inicial ("cero") por partición

In [41]:
suma_con_fold = numeros.fold(0, lambda a, b: a + b)
print("Suma con fold:", suma_con_fold)

Suma con fold: 5050


## `aggregate`: la acción más general y potente

Permite combinar elementos dentro de cada partición con una función,
y luego combinar los resultados parciales de las particiones con
otra función (posiblemente distinta), devolviendo un tipo de dato
diferente al de los elementos del RDD.

Ejemplo: calcular (suma, conteo) para poder obtener el promedio.

In [42]:
valor_cero = (0, 0)  # (suma_parcial, conteo_parcial)

def combinar_particion(acumulador, valor):
    suma, conteo = acumulador
    return (suma + valor, conteo + 1)

def combinar_particiones(acc1, acc2):
    return (acc1[0] + acc2[0], acc1[1] + acc2[1])

suma_total, conteo_total = numeros.aggregate(
    valor_cero, combinar_particion, combinar_particiones
)
print(f"Promedio = {suma_total} / {conteo_total} = {suma_total / conteo_total}")

Promedio = 5050 / 100 = 50.5


## `foreach`: ejecutar algo por cada elemento sin traer nada al driver

Se ejecuta en los ejecutores, no en el driver. No sirve para acumular
resultados en variables normales de Python (cada ejecutor tiene su
propia copia); para eso existen los *acumuladores* (notebook 7).

In [43]:
def imprimir_si_multiplo_de_20(x):
    if x % 20 == 0:
        print(f"[ejecutor] {x} es múltiplo de 20")

numeros.foreach(imprimir_si_multiplo_de_20)
print("foreach no imprime nada aquí en el driver: los prints ocurren en los ejecutores")

foreach no imprime nada aquí en el driver: los prints ocurren en los ejecutores


## `saveAsTextFile`: persistir el resultado a disco

Escribe **una carpeta** con un archivo `part-00000`, `part-00001`, ...
por cada partición. Esto es intencional: en un clúster real cada
ejecutor escribe su propia partición de forma independiente y en
paralelo, sin coordinarse con los demás.

In [44]:
import shutil
ruta_salida = "salida_pares"
shutil.rmtree(ruta_salida, ignore_errors=True)

pares = numeros.filter(lambda x: x % 2 == 0)
print("Particiones del RDD a guardar:", pares.getNumPartitions())
pares.saveAsTextFile(ruta_salida)

import os
print("Archivos generados:", sorted(os.listdir(ruta_salida)))

Particiones del RDD a guardar: 8
Archivos generados: ['._SUCCESS.crc', '.part-00000.crc', '.part-00001.crc', '.part-00002.crc', '.part-00003.crc', '.part-00004.crc', '.part-00005.crc', '.part-00006.crc', '.part-00007.crc', '_SUCCESS', 'part-00000', 'part-00001', 'part-00002', 'part-00003', 'part-00004', 'part-00005', 'part-00006', 'part-00007']


In [45]:
spark.stop()

---

# 5. RDDs de pares clave-valor (Pair RDDs)

Cuando cada elemento de un RDD es una tupla `(clave, valor)`, Spark
habilita un conjunto adicional de operaciones muy usadas en
procesamiento de Big Data: agregaciones por clave, joins, ordenar por
clave, etc. Esto es exactamente el patrón **map-reduce** clásico.

In [46]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("05-pares-clave-valor")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

## Construyendo un Pair RDD desde nuestro CSV de ventas

In [47]:
ventas_raw = sc.textFile("ventas.csv")
encabezado = ventas_raw.first()

ventas = (
    ventas_raw
    .filter(lambda linea: linea != encabezado)
    .map(lambda linea: linea.split(","))
)
ventas.take(2)

[['2026-01-05', 'Manizales', 'Cafe Premium', 'Bebidas', '120', '18500'],
 ['2026-01-05', 'Bogota', 'Cafe Premium', 'Bebidas', '300', '18500']]

Columnas: `fecha, ciudad, producto, categoria, unidades, precio_unitario`.
Construyamos un Pair RDD `(ciudad, unidades_vendidas)`.

In [48]:
ciudad_unidades = ventas.map(lambda fila: (fila[1], int(fila[4])))
ciudad_unidades.collect()

[('Manizales', 120),
 ('Bogota', 300),
 ('Medellin', 80),
 ('Manizales', 200),
 ('Cali', 150),
 ('Bogota', 90),
 ('Manizales', 60),
 ('Medellin', 210),
 ('Cali', 140),
 ('Bogota', 175),
 ('Manizales', 95),
 ('Medellin', 130)]

## `reduceByKey`: la agregación más importante de Big Data

Combina los valores que comparten la misma clave usando una función
asociativa. A diferencia de `groupByKey`, la combinación ocurre
**parcialmente dentro de cada partición antes de mezclar datos por la
red** (lo que se conoce como *combiner* en el modelo MapReduce), lo
cual reduce drásticamente el tráfico de red (*shuffle*).

In [49]:
unidades_por_ciudad = ciudad_unidades.reduceByKey(lambda a, b: a + b)
print(sorted(unidades_por_ciudad.collect()))

[('Bogota', 565), ('Cali', 290), ('Manizales', 475), ('Medellin', 420)]


## `groupByKey` vs `reduceByKey`: por qué importa la diferencia

`groupByKey` agrupa TODOS los valores de una clave en una lista antes
de agregar nada, lo que obliga a mover por la red todos los valores
individuales. `reduceByKey` agrega localmente primero y solo mueve
resultados parciales. Con datasets grandes, `reduceByKey` (o
`aggregateByKey`/`combineByKey`) es casi siempre preferible.

In [50]:
agrupado = ciudad_unidades.groupByKey()
# ResultIterable no es una lista directamente; hay que materializarla
for ciudad, valores in agrupado.collect():
    print(ciudad, "->", list(valores))

Manizales -> [120, 200, 60, 95]
Bogota -> [300, 90, 175]
Medellin -> [80, 210, 130]
Cali -> [150, 140]


## `sortByKey`: ordenar por la clave

In [51]:
print(unidades_por_ciudad.sortByKey().collect())
print(unidades_por_ciudad.sortByKey(ascending=False).collect())

[('Bogota', 565), ('Cali', 290), ('Manizales', 475), ('Medellin', 420)]
[('Medellin', 420), ('Manizales', 475), ('Cali', 290), ('Bogota', 565)]


## `keys()`, `values()`, `mapValues()`

`mapValues` transforma solo el valor de cada par, sin tocar la clave
(y sin re-particionar los datos, lo cual es más eficiente que un
`map` genérico cuando solo interesa el valor).

In [52]:
print("Solo claves:", unidades_por_ciudad.keys().collect())
print("Solo valores:", unidades_por_ciudad.values().collect())

en_cajas = unidades_por_ciudad.mapValues(lambda unidades: unidades // 12)
print("Unidades convertidas a 'cajas de 12':", en_cajas.collect())

Solo claves: ['Manizales', 'Bogota', 'Medellin', 'Cali']
Solo valores: [475, 565, 420, 290]
Unidades convertidas a 'cajas de 12': [('Manizales', 39), ('Bogota', 47), ('Medellin', 35), ('Cali', 24)]


## Ingresos totales por categoría (ejemplo más completo)

In [53]:
categoria_ingreso = ventas.map(
    lambda fila: (fila[3], int(fila[4]) * float(fila[5]))
)
ingreso_por_categoria = categoria_ingreso.reduceByKey(lambda a, b: a + b)
for categoria, ingreso in sorted(ingreso_por_categoria.collect()):
    print(f"{categoria}: ${ingreso:,.0f}")

Alimentos: $4,141,000
Bebidas: $16,187,500


## `join`: combinar dos Pair RDDs por clave

Igual que un `JOIN` en SQL, pero operando sobre RDDs. También existen
`leftOuterJoin`, `rightOuterJoin` y `fullOuterJoin`.

In [54]:
poblacion_ciudad = sc.parallelize([
    ("Manizales", 434_000),
    ("Bogota", 8_000_000),
    ("Medellin", 2_600_000),
    ("Cali", 2_300_000),
    ("Pereira", 480_000),  # ciudad que no aparece en las ventas
])

combinado = unidades_por_ciudad.join(poblacion_ciudad)
print("Inner join (solo ciudades presentes en ambos RDDs):")
for ciudad, (unidades, poblacion) in sorted(combinado.collect()):
    print(f"  {ciudad}: {unidades} unidades, población {poblacion:,}")

Inner join (solo ciudades presentes en ambos RDDs):
  Bogota: 565 unidades, población 8,000,000
  Cali: 290 unidades, población 2,300,000
  Manizales: 475 unidades, población 434,000
  Medellin: 420 unidades, población 2,600,000


In [55]:
left_join = unidades_por_ciudad.leftOuterJoin(poblacion_ciudad)
print("\nLeft outer join (todas las ciudades con ventas, aunque falte población):")
for ciudad, (unidades, poblacion) in sorted(left_join.collect()):
    print(f"  {ciudad}: {unidades} unidades, población {poblacion}")


Left outer join (todas las ciudades con ventas, aunque falte población):
  Bogota: 565 unidades, población 8000000
  Cali: 290 unidades, población 2300000
  Manizales: 475 unidades, población 434000
  Medellin: 420 unidades, población 2600000


## `countByKey`: acción que cuenta ocurrencias por clave

In [56]:
categoria_pais = ventas.map(lambda fila: (fila[3], 1))
print(dict(categoria_pais.countByKey()))

{'Bebidas': 5, 'Alimentos': 7}


In [57]:
spark.stop()

---

# 6. Persistencia (caché) y particionamiento

Dos temas de rendimiento que son centrales para trabajar bien con
Big Data en Spark: **cuándo reutilizar un RDD sin recalcularlo**, y
**cómo están distribuidos físicamente los datos entre particiones**.

In [58]:
import time
from pyspark.sql import SparkSession
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("06-persistencia-particionamiento")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

## El problema: recomputar el mismo RDD varias veces

Como los RDDs son perezosos, si usamos el mismo RDD "costoso" en dos
acciones distintas **sin persistirlo**, Spark recalcula todo el
linaje desde cero cada vez.

In [59]:
def transformacion_costosa(x):
    # simula una operación pesada
    time.sleep(0.0005)
    return x * 2

base = sc.parallelize(range(1, 3001), numSlices=8)
costoso = base.map(transformacion_costosa)

inicio = time.time()
total_1 = costoso.count()
print(f"1ra acción (sin caché): {time.time() - inicio:.2f}s -> count={total_1}")

inicio = time.time()
suma_1 = costoso.sum()
print(f"2da acción (sin caché, recalcula todo): {time.time() - inicio:.2f}s -> suma={suma_1}")

1ra acción (sin caché): 3.66s -> count=3000
2da acción (sin caché, recalcula todo): 1.16s -> suma=9003000


## La solución: `cache()` / `persist()`

`cache()` es un atajo para `persist(StorageLevel.MEMORY_ONLY)`.
Marca el RDD para que, la primera vez que se materialice por una
acción, sus particiones queden guardadas (en memoria, disco, o
ambos) para las siguientes acciones.

In [60]:
costoso_cacheado = base.map(transformacion_costosa).cache()

inicio = time.time()
total_2 = costoso_cacheado.count()  # aquí se calcula Y se guarda en caché
print(f"1ra acción (llena la caché): {time.time() - inicio:.2f}s -> count={total_2}")

inicio = time.time()
suma_2 = costoso_cacheado.sum()  # reutiliza la caché, no recalcula el map
print(f"2da acción (lee de caché): {time.time() - inicio:.2f}s -> suma={suma_2}")

1ra acción (llena la caché): 1.27s -> count=3000
2da acción (lee de caché): 0.24s -> suma=9003000


## Niveles de almacenamiento (`StorageLevel`)

`persist()` permite elegir explícitamente dónde y cómo se guardan las
particiones cacheadas:

| Nivel | Dónde | Notas |
|---|---|---|
| `MEMORY_ONLY` | Solo RAM | Si no cabe, las particiones que sobran se recalculan al necesitarse |
| `MEMORY_AND_DISK` | RAM, y disco si no alcanza la RAM | Buen default cuando el dataset es grande |
| `DISK_ONLY` | Solo disco | Más lento, pero no compite por RAM |
| `MEMORY_ONLY_2` / `MEMORY_AND_DISK_2` | Igual, pero con 2 réplicas | Tolerancia a fallos extra, más costo de memoria |

In [61]:
otro_rdd = sc.parallelize(range(1, 100))
otro_rdd.persist(StorageLevel.MEMORY_AND_DISK)
otro_rdd.count()  # dispara el cálculo y la persistencia
print("Nivel de almacenamiento actual:", otro_rdd.getStorageLevel())

Nivel de almacenamiento actual: Disk Memory Serialized 1x Replicated


Cuando ya no se necesita un RDD cacheado, es buena práctica liberar
la memoria con `unpersist()`.

In [62]:
otro_rdd.unpersist()
costoso_cacheado.unpersist()

PythonRDD[3] at RDD at PythonRDD.scala:57

## Particionamiento: cuántos "trozos" y en qué orden físico

El número de particiones determina el paralelismo máximo real del
trabajo: si tienes 4 núcleos pero solo 1 partición, solo un núcleo
trabaja a la vez.

In [63]:
rdd_1_particion = sc.parallelize(range(1, 1001), numSlices=1)
rdd_8_particiones = sc.parallelize(range(1, 1001), numSlices=8)

print("Particiones caso 1:", rdd_1_particion.getNumPartitions())
print("Particiones caso 2:", rdd_8_particiones.getNumPartitions())

Particiones caso 1: 1
Particiones caso 2: 8


## `repartition` vs `coalesce`

- `repartition(n)` puede **aumentar o disminuir** el número de
  particiones, pero siempre hace un *shuffle* completo (mueve datos
  por la red entre todos los nodos) — costoso.
- `coalesce(n)` solo sirve para **disminuir** particiones, y evita el
  shuffle completo cuando es posible (junta particiones vecinas), por
  lo que suele ser más barato para reducir particiones, por ejemplo
  antes de escribir a disco.

In [64]:
mas_particiones = rdd_1_particion.repartition(6)
print("Tras repartition(6):", mas_particiones.getNumPartitions())

menos_particiones = rdd_8_particiones.coalesce(2)
print("Tras coalesce(2):", menos_particiones.getNumPartitions())

Tras repartition(6): 6
Tras coalesce(2): 2


## Particionamiento personalizado para Pair RDDs: `partitionBy`

Para RDDs de pares clave-valor se puede controlar explícitamente en
qué partición cae cada clave, usando un `partitionFunc`. Esto es
clave para optimizar operaciones repetidas de `join`/`reduceByKey`
sobre las mismas claves, evitando shuffles repetidos.

In [65]:
pares = sc.parallelize([(i, f"valor-{i}") for i in range(20)])

pares_particionado = pares.partitionBy(4, partitionFunc=lambda clave: clave % 4)

print("Particiones tras partitionBy:", pares_particionado.getNumPartitions())
for i, particion in enumerate(pares_particionado.glom().collect()):
    claves = [clave for clave, _ in particion]
    print(f"  Partición {i}: claves = {claves}")

Particiones tras partitionBy: 4
  Partición 0: claves = [0, 4, 8, 12, 16]
  Partición 1: claves = [1, 5, 9, 13, 17]
  Partición 2: claves = [2, 6, 10, 14, 18]
  Partición 3: claves = [3, 7, 11, 15, 19]


Todas las claves `0, 4, 8, 12, 16` (residuo 0 al dividir por 4) caen
en la misma partición. Si luego hacemos varios `reduceByKey` o
`join` con RDDs particionados de la misma forma, Spark puede evitar
mover datos por la red porque ya sabe que las claves coincidentes
están en el mismo nodo (esto se llama *co-particionamiento*).

In [66]:
spark.stop()

---

# 7. Variables compartidas: Broadcast y Acumuladores

Cuando Spark ejecuta una función (como el lambda de un `map`) en los
ejecutores, por defecto **copia** cualquier variable externa que la
función use hacia cada tarea. Esto es ineficiente para datos grandes
y no sirve para acumular resultados desde los ejecutores de vuelta al
driver. Para esos dos problemas existen las variables compartidas.

In [67]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("07-variables-compartidas")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

## Broadcast variables: compartir datos de solo lectura de forma eficiente

Un caso típico: tenemos una tabla pequeña de referencia (por ejemplo,
un diccionario ciudad → departamento) que queremos usar dentro de un
`map` sobre un RDD grande. Sin broadcast, Spark serializaría y
enviaría ese diccionario **una vez por cada tarea**. Con
`sc.broadcast`, se envía **una sola vez por ejecutor** y se guarda en
caché ahí, sin importar cuántas tareas corran en ese ejecutor.

In [69]:
departamento_por_ciudad = {
    "Manizales": "Caldas",
    "Bogota": "Cundinamarca / D.C.",
    "Medellin": "Antioquia",
    "Cali": "Valle del Cauca",
}

# Se distribuye una sola vez a cada ejecutor
bc_departamentos = sc.broadcast(departamento_por_ciudad)

ventas_raw = sc.textFile("ventas.csv")
encabezado = ventas_raw.first()
ciudades_rdd = (
    ventas_raw
    .filter(lambda linea: linea != encabezado)
    .map(lambda linea: linea.split(",")[1])
)

# dentro del lambda accedemos al diccionario vía .value
con_departamento = ciudades_rdd.map(
    lambda ciudad: (ciudad, bc_departamentos.value.get(ciudad, "desconocido"))
).distinct()

print(sorted(con_departamento.collect()))

[('Bogota', 'Cundinamarca / D.C.'), ('Cali', 'Valle del Cauca'), ('Manizales', 'Caldas'), ('Medellin', 'Antioquia')]


Cuando ya no se necesita, conviene liberar la copia broadcast de la
memoria de los ejecutores con `.unpersist()` (o `.destroy()` si no se
volverá a usar nunca más).

In [70]:
bc_departamentos.unpersist()

## Acumuladores: agregar valores desde los ejecutores hacia el driver

Los acumuladores son variables de "solo escritura" desde los
ejecutores (cada tarea puede sumarles algo con `.add()`) y de
"solo lectura" desde el driver (con `.value`). Sirven para cosas como
contar cuántos registros fueron inválidos durante un procesamiento
distribuido, sin tener que hacer una acción adicional solo para eso.

In [71]:
contador_invalidos = sc.accumulator(0)
suma_unidades = sc.accumulator(0)

def procesar_fila(fila):
    global contador_invalidos, suma_unidades
    try:
        unidades = int(fila[4])
        suma_unidades.add(unidades)
        return unidades
    except (ValueError, IndexError):
        contador_invalidos.add(1)
        return 0

ventas_filas = (
    ventas_raw
    .filter(lambda linea: linea != encabezado)
    .map(lambda linea: linea.split(","))
)

# foreach es una acción: aquí sí se ejecuta el recorrido completo
ventas_filas.foreach(procesar_fila)

print("Filas inválidas encontradas:", contador_invalidos.value)
print("Suma total de unidades (vía acumulador):", suma_unidades.value)

Filas inválidas encontradas: 0
Suma total de unidades (vía acumulador): 1750


##  Advertencia importante sobre los acumuladores

Los acumuladores se garantizan correctos solo dentro de **acciones**.
Si se actualizan dentro de una **transformación** (como un `map`) y
esa transformación se recalcula más de una vez (por ejemplo, por
falta de caché, o por una re-ejecución tras un fallo), el acumulador
puede contar de más. Por eso el patrón seguro es actualizarlos dentro
de funciones usadas en acciones como `foreach`, como hicimos arriba,
y no depender de ellos dentro de un `map` cuyo resultado no se
consume con una acción inmediatamente.

In [72]:
spark.stop()

---

# 8. Caso integrador: WordCount y análisis de ventas

WordCount es el "hola mundo" de Big Data: contar cuántas veces
aparece cada palabra en un texto. Es el ejemplo perfecto para ver
**todo el pipeline de RDDs trabajando junto**: creación, transformación
encadenada y acción final. Después resolvemos un segundo caso, más
cercano a un problema real de negocio, usando el CSV de ventas.

In [73]:
import re
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("08-caso-wordcount")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

## Paso a paso: WordCount clásico

El patrón es siempre el mismo:

1. `textFile` → un RDD de líneas.
2. `flatMap` → separar cada línea en palabras (aplanando).
3. `map` → convertir cada palabra en un par `(palabra, 1)`.
4. `reduceByKey` → sumar los conteos por palabra.

In [74]:
lineas = sc.textFile("quijote_fragmento.txt")


def limpiar_y_tokenizar(linea):
    linea = linea.lower()
    linea = re.sub(r"[^a-záéíóúñü\s]", "", linea)
    return linea.split()


conteo_palabras = (
    lineas
    .flatMap(limpiar_y_tokenizar)
    .map(lambda palabra: (palabra, 1))
    .reduceByKey(lambda a, b: a + b)
)

Hasta aquí no se ha ejecutado nada real (todo perezoso). Disparemos
el cómputo y veamos las palabras más frecuentes.

In [75]:
top_10 = conteo_palabras.takeOrdered(10, key=lambda par: -par[1])
for palabra, veces in top_10:
    print(f"{palabra:15s} {veces}")

de              14
los             7
y               6
que             5
un              3
no              3
una             3
mas             3
las             3
su              3


`takeOrdered(n, key=...)` es una acción muy conveniente: ordena por
la clave dada y trae solo los `n` primeros, sin necesidad de traer
(`collect`) todo el RDD ordenado al driver.

## Versión "una sola expresión" (estilo funcional típico de Spark)

In [76]:
resultado_compacto = (
    sc.textFile("quijote_fragmento.txt")
    .flatMap(limpiar_y_tokenizar)
    .map(lambda p: (p, 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda par: par[1] > 1)  # solo palabras que se repiten
    .sortBy(lambda par: -par[1])
)
resultado_compacto.collect()

[('de', 14),
 ('los', 7),
 ('y', 6),
 ('que', 5),
 ('un', 3),
 ('no', 3),
 ('una', 3),
 ('mas', 3),
 ('las', 3),
 ('su', 3),
 ('en', 3),
 ('la', 2),
 ('lo', 2),
 ('rocin', 2),
 ('el', 2),
 ('con', 2)]

## Segundo caso: reporte de ventas por ciudad y categoría

Ahora resolvamos algo más parecido a un problema de negocio real:
a partir del CSV de ventas, calcular el ingreso total por ciudad,
encontrar la ciudad con mayor ingreso, y el producto más vendido en
unidades — todo con operaciones de RDD.

In [77]:
ventas_raw = sc.textFile("ventas.csv")
encabezado = ventas_raw.first()

ventas = (
    ventas_raw
    .filter(lambda linea: linea != encabezado)
    .map(lambda linea: linea.split(","))
    .map(lambda f: {
        "fecha": f[0],
        "ciudad": f[1],
        "producto": f[2],
        "categoria": f[3],
        "unidades": int(f[4]),
        "precio_unitario": float(f[5]),
    })
)
ventas.cache()  # lo vamos a usar varias veces abajo
ventas.take(2)

[{'fecha': '2026-01-05',
  'ciudad': 'Manizales',
  'producto': 'Cafe Premium',
  'categoria': 'Bebidas',
  'unidades': 120,
  'precio_unitario': 18500.0},
 {'fecha': '2026-01-05',
  'ciudad': 'Bogota',
  'producto': 'Cafe Premium',
  'categoria': 'Bebidas',
  'unidades': 300,
  'precio_unitario': 18500.0}]

In [78]:
ingreso_por_ciudad = (
    ventas
    .map(lambda v: (v["ciudad"], v["unidades"] * v["precio_unitario"]))
    .reduceByKey(lambda a, b: a + b)
)

ciudad_top, ingreso_top = ingreso_por_ciudad.reduce(
    lambda a, b: a if a[1] > b[1] else b
)
print(f"Ciudad con mayor ingreso: {ciudad_top} (${ingreso_top:,.0f})")

print("\nIngreso por ciudad (ordenado de mayor a menor):")
for ciudad, ingreso in ingreso_por_ciudad.sortBy(lambda par: -par[1]).collect():
    print(f"  {ciudad:12s} ${ingreso:>12,.0f}")

Ciudad con mayor ingreso: Bogota ($6,933,000)

Ingreso por ciudad (ordenado de mayor a menor):
  Bogota       $   6,933,000
  Medellin     $   5,157,000
  Manizales    $   5,029,500
  Cali         $   3,209,000


In [79]:
unidades_por_producto = (
    ventas
    .map(lambda v: (v["producto"], v["unidades"]))
    .reduceByKey(lambda a, b: a + b)
)

producto_top, unidades_top = unidades_por_producto.reduce(
    lambda a, b: a if a[1] > b[1] else b
)
print(f"Producto más vendido en unidades: {producto_top} ({unidades_top} unidades)")

Producto más vendido en unidades: Cafe Premium (875 unidades)


In [80]:
ventas.unpersist()
spark.stop()

---

# 9. RDD vs DataFrame: ¿cuándo usar cada uno?

Este notebook cierra el libro comparando directamente el mismo
problema resuelto con RDDs y con DataFrames, para que quede claro qué
se gana y qué se pierde con cada abstracción.

In [81]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("09-rdd-vs-dataframe")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

## El mismo problema, dos maneras: ingreso total por categoría

### Con RDD

In [82]:
ventas_raw = sc.textFile("ventas.csv")
encabezado = ventas_raw.first()

ingreso_por_categoria_rdd = (
    ventas_raw
    .filter(lambda linea: linea != encabezado)
    .map(lambda linea: linea.split(","))
    .map(lambda f: (f[3], int(f[4]) * float(f[5])))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda par: -par[1])
)
ingreso_por_categoria_rdd.collect()

[('Bebidas', 16187500.0), ('Alimentos', 4141000.0)]

### Con DataFrame

In [84]:
df_ventas = spark.read.csv("ventas.csv", header=True, inferSchema=True)

ingreso_por_categoria_df = (
    df_ventas
    .withColumn("ingreso", F.col("unidades") * F.col("precio_unitario"))
    .groupBy("categoria")
    .agg(F.sum("ingreso").alias("ingreso_total"))
    .orderBy(F.desc("ingreso_total"))
)
ingreso_por_categoria_df.show()

+---------+-------------+
|categoria|ingreso_total|
+---------+-------------+
|  Bebidas|     16187500|
|Alimentos|      4141000|
+---------+-------------+



## Comparación

| Aspecto | RDD | DataFrame |
|---|---|---|
| Nivel de abstracción | Bajo (colecciones distribuidas genéricas) | Alto (tablas con esquema, como SQL) |
| Optimización automática | No — el plan lo escribes tú a mano | Sí — el optimizador **Catalyst** reescribe el plan de ejecución |
| Rendimiento típico | Menor (más overhead de serialización en Python) | Mayor (ejecución en JVM, código generado) |
| Verificación de tipos/columnas | En tiempo de ejecución (son tuplas/objetos genéricos) | En el esquema, con validaciones más tempranas |
| Control fino de particionamiento | Total (`partitionBy`, `mapPartitions`, etc.) | Limitado / indirecto |
| Datos no tabulares (grafos, texto libre, objetos anidados complejos) | Natural | Incómodo o requiere UDFs |
| Curva de aprendizaje | Requiere pensar en map/reduce | Similar a SQL / pandas |

## Guía práctica de decisión

**Usa DataFrame (casi siempre) cuando:**
- Los datos tienen o pueden tener un esquema tabular (filas y columnas).
- Necesitas agregaciones tipo SQL, joins entre tablas, lectura de
  Parquet/CSV/JSON/bases de datos.
- Quieres que Spark optimice automáticamente el plan de ejecución.

**Usa RDD cuando:**
- Necesitas control explícito y detallado del particionamiento físico
  de los datos (por ejemplo, algoritmos iterativos tipo PageRank).
- Los datos no son tabulares (grafos, árboles, texto no estructurado
  antes de parsearlo, objetos Python complejos).
- Estás implementando una transformación tan específica que no se
  puede expresar razonablemente con las funciones de `pyspark.sql.functions`
  ni con una UDF simple.
- Estás aprendiendo cómo funciona Spark internamente — los DataFrames
  se compilan, en última instancia, a operaciones sobre RDDs.

En la práctica, en pipelines modernos de datos, se empieza casi
siempre con DataFrames, y se cae a nivel de RDD (accesible siempre
vía `df.rdd`) solo cuando se topa con una de las excepciones de
arriba.

In [85]:
# De DataFrame a RDD y viceversa: son completamente interoperables
rdd_desde_df = df_ventas.rdd
print("Primer registro como Row (RDD subyacente):", rdd_desde_df.first())

df_desde_rdd = ingreso_por_categoria_rdd.toDF(["categoria", "ingreso_total"])
df_desde_rdd.show()

Primer registro como Row (RDD subyacente): Row(fecha=datetime.date(2026, 1, 5), ciudad='Manizales', producto='Cafe Premium', categoria='Bebidas', unidades=120, precio_unitario=18500)
+---------+-------------+
|categoria|ingreso_total|
+---------+-------------+
|  Bebidas|    1.61875E7|
|Alimentos|    4141000.0|
+---------+-------------+



In [86]:
spark.stop()